# Лабораторная работа 1 — Методы улучшения контраста изображений (вариант 15)

Исходное изображение: `lectures/3 Методы улучшения контраста изображений/lab1/image_v1-15.png`.

Цель: применить базовые методы улучшения контраста и сравнить результаты по ряду метрик качества/контраста.


In [ ]:
import os
import numpy as np
import cv2
from PIL import Image
from IPython.display import display
import matplotlib.pyplot as plt
from skimage import exposure

IMAGE_PATH = "/Users/23108022/Documents/repositories/mephi-computer-vision-2025/lectures/3 Методы улучшения контраста изображений/lab1/image_v1-15.png"
assert os.path.exists(IMAGE_PATH), "Image not found"

# read as BGR then convert to RGB for correct display
bgr = cv2.imread(IMAGE_PATH, cv2.IMREAD_COLOR)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)

print(f"shape={rgb.shape}, dtype={rgb.dtype}, min={rgb.min()}, max={rgb.max()}, mean={rgb.mean():.2f}")
display(Image.fromarray(rgb))


In [ ]:
def show_hist(img, title="histogram", cmap=None):
    plt.figure(figsize=(5,3))
    if img.ndim == 2:
        plt.hist(img.ravel(), bins=256, range=[0,256], color='gray')
    else:
        colors = ('r','g','b')
        for c, col in enumerate(colors):
            hist = cv2.calcHist([img],[c],None,[256],[0,256])
            plt.plot(hist, color=col)
            plt.xlim([0,256])
    plt.title(title)
    plt.tight_layout()
    plt.show()

show_hist(gray, title="Grayscale histogram (original)")


In [ ]:
# 1) Линейное растяжение гистограммы (min-max)
mn, mx = np.min(gray), np.max(gray)
lin = ((gray - mn) * (255.0 / max(1, (mx - mn)))).clip(0,255).astype(np.uint8)

# 2) Гамма-коррекция
gamma = 0.6  # подберите при необходимости
norm = gray.astype(np.float32) / 255.0
gamma_img = np.power(norm, gamma)
gamma_img = (gamma_img * 255.0).clip(0,255).astype(np.uint8)

# 3) Выравнивание гистограммы (global HE)
he = cv2.equalizeHist(gray)

# 4) CLAHE
clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8))
clahe_img = clahe.apply(gray)

# Визуализация
row1 = cv2.hconcat([
    cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(lin, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(gamma_img, cv2.COLOR_GRAY2BGR)
])
row2 = cv2.hconcat([
    cv2.cvtColor(he, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(clahe_img, cv2.COLOR_GRAY2BGR),
    cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
])
vis = cv2.vconcat([row1, row2])

# показать
display(Image.fromarray(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)))

# Сохранить результаты
OUT_DIR = "/Users/23108022/Documents/repositories/mephi-computer-vision-2025/lessons/lesson2/lab1_outputs"
os.makedirs(OUT_DIR, exist_ok=True)
Image.fromarray(gray).save(os.path.join(OUT_DIR, "01_original_gray.png"))
Image.fromarray(lin).save(os.path.join(OUT_DIR, "02_linear_stretch.png"))
Image.fromarray(gamma_img).save(os.path.join(OUT_DIR, "03_gamma_0_6.png"))
Image.fromarray(he).save(os.path.join(OUT_DIR, "04_equalize_hist.png"))
Image.fromarray(clahe_img).save(os.path.join(OUT_DIR, "05_CLAHE.png"))
print(f"Saved to {OUT_DIR}")


In [ ]:
# Метрики контраста и качества
import math
from skimage.metrics import structural_similarity as ssim

def mich_contrast(img):
    amax = float(img.max())
    amin = float(img.min())
    if amax + amin == 0:
        return 0.0
    return (amax - amin) / (amax + amin)

def global_contrast(img):
    return (float(img.max()) - float(img.min())) / 255.0

def rms_contrast(img):
    return float(img.std())

variants = {
    "original": gray,
    "linear": lin,
    "gamma": gamma_img,
    "equalize": he,
    "clahe": clahe_img,
}

rows = []
for name, im in variants.items():
    rows.append((
        name,
        mich_contrast(im),
        global_contrast(im),
        rms_contrast(im),
        ssim(gray, im, data_range=255)
    ))

import pandas as pd
pd.DataFrame(rows, columns=["image","C_Michelson","C_global","C_rms","SSIM_vs_original"])


In [ ]:
# Гистограммы до/после
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.ravel()
for ax, (name, im) in zip(axes, [("original", gray), ("linear", lin), ("equalize", he), ("clahe", clahe_img)]):
    ax.hist(im.ravel(), bins=256, range=(0,256), color='black')
    ax.set_title(name)
plt.tight_layout()
plt.show()


In [ ]:
# Экспорт таблицы метрик
import pandas as pd
metrics_df = pd.DataFrame(rows, columns=["image","C_Michelson","C_global","C_rms","SSIM_vs_original"]) 
metrics_csv = os.path.join(OUT_DIR, "metrics.csv")
metrics_df.to_csv(metrics_csv, index=False)
metrics_df


### Выводы

- Линейное растяжение максимально использует динамический диапазон, повышая `C_global`.
- Гамма-коррекция (γ=0.6) усиливает тени; можно варьировать γ∈[0.4,1.8] под задачу.
- Глобальное выравнивание гистограммы повышает контраст, но может усиливать шум/артефакты.
- CLAHE дает более естественный результат на неоднородных сценах за счет локальной адаптации.
- В таблице `metrics.csv` сохранены значения `C_Michelson`, `C_global`, `C_rms`, `SSIM_vs_original`.
